# Multi-model LLM-as-Judge — US NLG Conditions

Evaluates all seven US NLG pipeline conditions with three judges:
**GPT-5**, **Claude Sonnet 3.7**, and **Gemini 2.5 Pro**.

- GPT-5 results are **loaded from existing files** — no re-run.
- Claude and Gemini are computed fresh and cached.

Outputs → `results/validation/llm_judge_multimodel/`

In [ ]:
from pathlib import Path
import statistics as stats
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Financial-D2T-Agent":
    PROJECT_ROOT = PROJECT_ROOT / "Financial-D2T-Agent"

RESULTS_ROOT = PROJECT_ROOT / "results"

CATEGORIES = [
    "nlg_brazilian_manager/e2e",
    "nlg_brazilian_manager/default",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/no_orchestrator_no_guardrail_no_finalizer",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/no_orchestrator_no_finalizer",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/no_guardrail_no_finalizer",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/e2e",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/default_old",
    "nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/default",
    "nlg/final_report2025_us/gpt-5/workflow_False/openai/gpt-5/en/e2e",
    "nlg/final_report2025_us/gpt-5/workflow_False/openai/gpt-5/en/default_old",
    "nlg/final_report2025_us/gpt-5/workflow_False/openai/gpt-5/en/default",
]

try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt-5")
    def count_tokens(text: str) -> int:
        return len(enc.encode(text))
    tokenizer_name = "tiktoken:gpt-5"
except Exception:
    def count_tokens(text: str) -> int:
        return len(text.split())
    tokenizer_name = "fallback:whitespace_words"

rows = []

for category in CATEGORIES:
    folder = RESULTS_ROOT / category
    txt_files = sorted(folder.glob("*.txt"))

    token_counts = []
    char_counts = []
    word_counts = []

    for path in txt_files:
        text = path.read_text(encoding="utf-8", errors="replace").strip()
        token_counts.append(count_tokens(text))
        char_counts.append(len(text))
        word_counts.append(len(text.split()))

    def safe_mean(values):
        return stats.mean(values) if values else 0

    def safe_median(values):
        return stats.median(values) if values else 0

    def safe_stdev(values):
        return stats.stdev(values) if len(values) > 1 else 0

    rows.append({
        "category": category,
        "folder_exists": folder.exists(),
        "n_files": len(txt_files),
        "tokenizer": tokenizer_name,
        "tokens_total": sum(token_counts),
        "tokens_mean": round(safe_mean(token_counts), 2),
        "tokens_median": round(safe_median(token_counts), 2),
        "tokens_min": min(token_counts) if token_counts else 0,
        "tokens_max": max(token_counts) if token_counts else 0,
        "tokens_stdev": round(safe_stdev(token_counts), 2),
        "words_mean": round(safe_mean(word_counts), 2),
        "Words (min)": min(word_counts) if word_counts else 0,
        "Words (max)": max(word_counts) if word_counts else 0,
        "Words (SD)": round(stats.stdev(word_counts), 2) if len(word_counts) > 1 else 0,
        "chars_mean": round(safe_mean(char_counts), 2),
        "chars (min)": min(char_counts) if char_counts else 0,
        "chars (max)": max(char_counts) if char_counts else 0,
        "chars (SD)": round(stats.stdev(char_counts), 2) if len(char_counts) > 1 else 0,
    })

token_stats_df = pd.DataFrame(rows)
token_stats_df


,category,folder_exists,n_files,tokenizer,tokens_total,tokens_mean,tokens_median,tokens_min,tokens_max,tokens_stdev,words_mean,Words (min),Words (max),Words (SD),chars_mean,chars (min),chars (max),chars (SD)
0,nlg_brazilian_manager/e2e,True,24,tiktoken:gpt-5,127259,5302.46,5150.5,4218,6446,816.96,2435.88,2037,2996,281.22,15297.67,12752,18145,1700.27
1,nlg_brazilian_manager/default,True,24,tiktoken:gpt-5,136192,5674.67,5160.0,4411,9036,1297.92,2722.58,2157,4145,539.90,17089.71,13741,24842,3162.28
2,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,75036,5359.71,5377.5,4768,6054,427.74,2636.57,2339,2928,178.03,16869.21,15246,18670,1054.64
3,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,72610,5186.43,5066.0,4665,5787,358.73,2573.64,2326,2866,159.68,16369.86,14908,18419,1026.18
4,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,77174,5512.43,5413.5,4358,6394,591.59,2644.57,2147,2987,256.67,16925.50,13782,19126,1626.92
5,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,62451,4460.79,4425.5,3891,4853,288.84,2365.93,2152,2550,132.43,14932.21,13722,16052,773.93
6,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,66756,4768.29,4926.0,3054,6003,856.21,2436.07,1615,2972,417.32,15427.71,10122,19319,2673.75
7,nlg/final_report2025_us/gpt-5/workflow_True/op...,True,14,tiktoken:gpt-5,68387,4884.79,4752.5,4334,5633,421.67,2402.21,2152,2773,179.37,15354.29,14048,18091,1238.60
8,nlg/final_report2025_us/gpt-5/workflow_False/o...,True,14,tiktoken:gpt-5,60892,4349.43,4445.5,3798,4707,287.47,2324.29,2158,2477,116.35,14769.79,13705,15721,708.07
9,nlg/final_report2025_us/gpt-5/workflow_False/o...,True,14,tiktoken:gpt-5,74600,5328.57,5372.0,4518,6532,516.37,2794.00,2378,3395,253.21,17610.07,15116,21982,1684.31


In [ ]:
out_path = PROJECT_ROOT / "results" / "nlg_token_statistics_by_category.csv"
token_stats_df.to_csv(out_path, index=False)
out_path


PosixPath('/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/nlg_token_statistics_by_category.csv')

In [ ]:
from __future__ import annotations

import json, os, re, sys, time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field
from scipy import stats as scipy_stats

load_dotenv(Path.home() / ".env")
load_dotenv()

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Financial-D2T-Agent":
    PROJECT_ROOT = PROJECT_ROOT / "Financial-D2T-Agent"

sys.path.insert(0, str(PROJECT_ROOT))
from load_data import build_multi_stock_prompt_context, load_generation_samples

# ── Conditions to evaluate ────────────────────────────────────────────────────
CONDITIONS: list[tuple[bool, str]] = [
    (False, "default"),
    (False, "e2e"),
    (True,  "default"),
    (True,  "e2e"),
    (True,  "no_orchestrator_no_guardrail_no_finalizer"),
    (True,  "no_orchestrator_no_finalizer"),
    (True,  "no_guardrail_no_finalizer"),
]

# ── Judge configuration ───────────────────────────────────────────────────────
GEMINI_MODEL_ID = os.getenv("GEMINI_25_AIXPLAIN_MODEL_ID",
                            os.getenv("AIXPLAIN_GEMINI_MODEL_ID", "google/gemini-2.5-pro/google"))
CLAUDE_MODEL    = os.getenv("CLAUDE_JUDGE_MODEL", "claude-3-7-sonnet-latest")

JUDGES = {
    "gpt5":            {"label": "GPT-5",            "enabled": bool(os.getenv("OPENAI_API_KEY"))},
    "claude_sonnet_37":{"label": "Claude Sonnet 3.7","enabled": bool(os.getenv("ANTHROPIC_API_KEY")),
                        "model": CLAUDE_MODEL},
    "gemini_25":       {"label": "Gemini 2.5 Pro",   "enabled": bool(os.getenv("AIXPLAIN_API_KEY") or os.getenv("TEAM_API_KEY")),
                        "model": GEMINI_MODEL_ID},
}

OVERWRITE    = False   # True to re-score already-saved files
MAX_RETRIES  = 3
DIMENSIONS   = ["No-Omissions", "No-Additions", "Grammaticality", "Coherence", "Fluency"]
SCORE_COLS   = [f"{d.lower().replace('-','_')}_score" for d in DIMENSIONS]

# ── Output paths ──────────────────────────────────────────────────────────────
RESULTS_ROOT = PROJECT_ROOT / "results"
OUT_ROOT     = RESULTS_ROOT / "validation" / "llm_judge_multimodel"
GPT5_ROOT    = RESULTS_ROOT / "validation" / "llm_judge" / "us" / "gpt-5"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

pd.DataFrame([{"judge": k, **{x: v[x] for x in ("label","enabled")}}
              for k, v in JUDGES.items()])

In [ ]:
# ── Schema ────────────────────────────────────────────────────────────────────

class DimensionScore(BaseModel):
    Justification: str = Field(min_length=1)
    Score: int = Field(ge=1, le=5)

class JudgeScorecard(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    no_omissions:   DimensionScore = Field(alias="No-Omissions")
    no_additions:   DimensionScore = Field(alias="No-Additions")
    grammaticality: DimensionScore = Field(alias="Grammaticality")
    coherence:      DimensionScore = Field(alias="Coherence")
    fluency:        DimensionScore = Field(alias="Fluency")

# ── Judge instructions — two variants, identical criteria ─────────────────────
# JUDGE_INSTRUCTIONS_EN   : English — for US conditions
# JUDGE_INSTRUCTIONS_PTBR : Brazilian Portuguese — for BR-PT conditions
# JSON output keys are the same in both so JudgeScorecard parses either.

JUDGE_INSTRUCTIONS_EN = """You are evaluating how well a Generated Report realises a given Input Bundle for a financial data-to-text task.

Your task:
1. Read the Input Bundle and the Generated Report carefully.
2. For each Dimension below, assign a score from 1 (lowest) to 5 (highest).
3. Give a short justification (1–2 sentences) per Dimension.
4. Return only a single JSON object in the exact format below. No extra text.

Dimensions:
No-Omissions: Degree to which ALL structured data fields in the Input Bundle appear in the report. Do not penalise missing year-over-year comparisons or growth rates that appear only in justification text and not in the structured data fields.
No-Additions: Degree to which the report contains ONLY information from the Input Bundle. Report metadata and inferences directly derivable from the figures are permitted. Only penalise figures or facts not traceable to the bundle.
Grammaticality: Degree to which the report is grammatically correct (form only).
Coherence: Degree to which the report is well-structured and logically organised (meaning only).
Fluency: Degree to which the report reads smoothly as professional financial prose.

Rules: scores must be integers 1–5; judge each Dimension independently; do not award 5 unless fully satisfied.

Return this exact JSON (no extra keys, no extra text):
{
  "No-Omissions":   {"Justification": "", "Score": 1},
  "No-Additions":   {"Justification": "", "Score": 1},
  "Grammaticality": {"Justification": "", "Score": 1},
  "Coherence":      {"Justification": "", "Score": 1},
  "Fluency":        {"Justification": "", "Score": 1}
}"""

JUDGE_INSTRUCTIONS_PTBR = """Você está avaliando o grau em que um Relatório Gerado realiza um Conjunto de Entrada em uma tarefa de geração de texto a partir de dados financeiros.

Sua tarefa:
1. Leia o Conjunto de Entrada e o Relatório Gerado com atenção.
2. Para cada Dimensão abaixo, atribua uma pontuação de 1 (mínimo) a 5 (máximo).
3. Forneça uma justificativa breve (1–2 frases) por Dimensão.
4. Retorne apenas um único objeto JSON no formato exato abaixo. Sem texto adicional.

Dimensões:
No-Omissions: Grau em que TODOS os campos de dados estruturados do Conjunto de Entrada aparecem no relatório. Não penalize a ausência de comparações ano a ano ou taxas de crescimento que aparecem apenas no texto de justificativa e não nos campos de dados estruturados.
No-Additions: Grau em que o relatório contém APENAS informações do Conjunto de Entrada. Metadados do relatório e inferências diretamente deriváveis dos dados são permitidos. Penalize apenas dados ou fatos não rastreáveis ao conjunto.
Grammaticality: Grau em que o relatório está gramaticalmente correto (somente forma).
Coherence: Grau em que o relatório está bem estruturado e logicamente organizado (somente significado).
Fluency: Grau em que o relatório é lido de forma fluida como prosa financeira profissional.

Regras: as pontuações devem ser inteiros de 1 a 5; avalie cada Dimensão de forma independente; não atribua 5 a menos que a condição esteja plenamente satisfeita.

Retorne este JSON exato (sem chaves extras, sem texto adicional):
{
  "No-Omissions":   {"Justification": "", "Score": 1},
  "No-Additions":   {"Justification": "", "Score": 1},
  "Grammaticality": {"Justification": "", "Score": 1},
  "Coherence":      {"Justification": "", "Score": 1},
  "Fluency":        {"Justification": "", "Score": 1}
}"""

INSTRUCTIONS_BY_LANG = {"en": JUDGE_INSTRUCTIONS_EN, "pt_br": JUDGE_INSTRUCTIONS_PTBR}

# US conditions are English; keep JUDGE_INSTRUCTIONS pointing to EN for existing call_* cells
JUDGE_INSTRUCTIONS = JUDGE_INSTRUCTIONS_EN

print("Schema and instructions ready.")
print(f"  EN   : {len(JUDGE_INSTRUCTIONS_EN)} chars")
print(f"  PT-BR: {len(JUDGE_INSTRUCTIONS_PTBR)} chars")

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────

def nlg_dir(sr: bool, wf: str) -> Path:
    return RESULTS_ROOT / "nlg" / "final_report2025_us" / "gpt-5" / f"workflow_{sr}" / "openai" / "gpt-5" / "en" / wf

def src_dir(sr: bool) -> Path:
    return RESULTS_ROOT / "final_report2025_us" / "gpt-5" / f"workflow_{sr}"

def gpt5_dir(sr: bool, wf: str) -> Path:
    return GPT5_ROOT / f"workflow_{sr}" / "openai" / "gpt-5" / "en" / wf

def judge_dir(judge: str, sr: bool, wf: str) -> Path:
    return OUT_ROOT / judge / f"workflow_{sr}" / "openai" / "gpt-5" / "en" / wf

def safe_slug(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", (text or "sample").strip())


def build_records(sr: bool, wf: str) -> list[dict[str, Any]]:
    samples = load_generation_samples(str(src_dir(sr)), dataset_kind="auto", min_stocks_per_month=1)
    idx = {str(s["sample_name"]): s for s in samples}

    rows = []
    for jp in sorted(nlg_dir(sr, wf).glob("*.json")):
        if "sequence_summary" in jp.name:
            continue
        p = json.loads(jp.read_text(encoding="utf-8"))
        m = p.get("sample_metadata") or {}
        name  = str(m.get("sample_name") or p.get("sample_name") or jp.stem)
        date  = str(m.get("analysis_date") or p.get("analysis_date") or "")[:10]
        text  = (p.get("generated_text") or p.get("final_response")
                 or (jp.with_suffix(".txt").read_text(encoding="utf-8")
                     if jp.with_suffix(".txt").exists() else "")).strip()
        rows.append({"name": name, "date": date, "text": text, "jp": jp})

    rows.sort(key=lambda r: r["date"])
    by_date = {r["date"]: r for r in rows}

    # infer coverage end date
    end = max((str(s.get("analysis_date",""))[:10] for s in idx.values()), default="")
    if end:
        ts = pd.to_datetime(end, errors="coerce")
        if not pd.isna(ts):
            end = (ts + pd.offsets.MonthEnd(0)).date().isoformat()

    records = []
    for row in rows:
        s = idx.get(row["name"])
        if not s:
            continue
        prev_d = str(s.get("previous_analysis_date") or "")[:10]
        prev_t = ("N/A" if not prev_d or prev_d not in by_date
                  else by_date[prev_d]["text"])
        if prev_t == "N/A" and isinstance(s.get("previous_report"), str):
            prev_t = s["previous_report"].strip() or "N/A"

        tickers = s.get("tickers") or []
        a_ts = pd.to_datetime(row["date"], errors="coerce")
        e_ts = pd.to_datetime(end, errors="coerce")
        horizon = str(max((e_ts.year-a_ts.year)*12+(e_ts.month-a_ts.month),0))                   if not (pd.isna(a_ts) or pd.isna(e_ts)) else ""

        records.append({
            "name": row["name"], "date": row["date"],
            "prev_date": prev_d or None,
            "text": row["text"], "prev_text": prev_t,
            "context": build_multi_stock_prompt_context(
                analysis_date=str(s.get("analysis_date","")),
                stock_rows=s.get("stocks",[]),
                previous_report=prev_t or "N/A",
            ),
            "meta": {
                "analysis_date": row["date"],
                "tickers": ", ".join(tickers) if isinstance(tickers,list) else str(tickers),
                "ticker_count": str(len(tickers) if isinstance(tickers,list) else ""),
                "end_date": end, "horizon_months": horizon,
            },
            "jp": row["jp"],
        })
    return records


def build_judge_input(r: dict[str, Any]) -> str:
    m = r["meta"]
    return (
        f"Sample name: {r['name']}\nAnalysis date: {r['date']}\n"
        f"Previous analysis date: {r['prev_date'] or 'N/A'}\n\n"
        f"Generation-prompt report metadata (treat as authoritative input facts, not additions):\n"
        f"Analysis month: {m['analysis_date']} | Coverage: {m['tickers']} "
        f"| Stocks: {m['ticker_count']} | Window end: {m['end_date']} "
        f"| Horizon: {m['horizon_months']} months\n\n"
        f"Current-month input bundle:\n{r['context']}\n\n"
        f"Previous report (continuity only):\n{r['prev_text']}\n\n"
        f"Generated report to score:\n{r['text']}"
    )


def load_gpt5_existing(sr: bool, wf: str) -> list[dict[str, Any]]:
    results = []
    for p in sorted(gpt5_dir(sr, wf).glob("*.json")):
        if p.name == "all_judgements.json":
            continue
        raw = json.loads(p.read_text(encoding="utf-8"))
        if "error" in raw or "scores" not in raw:
            continue
        results.append({"name": raw["sample_name"], "date": raw["analysis_date"],
                        "judge": "gpt5", "label": "GPT-5",
                        "attempt": raw.get("judge_attempt", 1), "scores": raw["scores"]})
    return results

print("Data loading functions ready.")

In [ ]:
# ── Judge API clients ─────────────────────────────────────────────────────────

def parse_scorecard(raw: str) -> JudgeScorecard:
    text = (raw or "").strip()
    candidates = [text]
    if text.startswith("```"):
        candidates.append(re.sub(r"^```(?:json)?\s*|\s*```$", "", text).strip())
    s, e = text.find("{"), text.rfind("}")
    if s >= 0 and e > s:
        candidates.append(text[s:e+1])
    last = None
    for c in dict.fromkeys(candidates):
        if not c: continue
        try:
            return JudgeScorecard.model_validate_json(c)
        except Exception as exc:
            last = exc
    raise ValueError(f"Cannot parse scorecard: {last}")


def call_gpt5(judge_input: str) -> JudgeScorecard:
    from openai import OpenAI
    resp = OpenAI().responses.parse(
        model="gpt-5", instructions=JUDGE_INSTRUCTIONS, input=judge_input,
        text_format=JudgeScorecard, reasoning={"effort": "high"},
        text={"verbosity": "low"}, store=False,
    )
    return resp.output_parsed or parse_scorecard(getattr(resp, "output_text", "") or "")


def call_claude(judge_input: str, model: str) -> JudgeScorecard:
    import anthropic
    resp = anthropic.Anthropic().messages.create(
        model=model, max_tokens=8192,
        system=JUDGE_INSTRUCTIONS,
        messages=[{"role": "user", "content": judge_input}],
    )
    return parse_scorecard(resp.content[0].text)


def call_gemini(judge_input: str, model_id: str) -> JudgeScorecard:
    from aixplain import Aixplain
    api_key = os.getenv("AIXPLAIN_API_KEY") or os.getenv("TEAM_API_KEY")
    model = Aixplain(api_key).Model.get(model_id)
    full  = f"{JUDGE_INSTRUCTIONS}\n\n{judge_input}"
    try:
        result = model.run(text=full, temperature=0.0, max_tokens=8192)
    except TypeError:
        result = model.run(text=full)
    raw = ""
    if hasattr(result, "data"):
        d = result.data
        raw = getattr(d, "output", "") or (d.get("output","") if isinstance(d,dict) else "")
    return parse_scorecard(raw or str(result))


def run_judge(record: dict[str, Any], judge: str, sr: bool, wf: str) -> dict[str, Any] | None:
    out = judge_dir(judge, sr, wf)
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{safe_slug(record['name'])}.json"
    if path.exists() and not OVERWRITE:
        return json.loads(path.read_text(encoding="utf-8"))
    cfg = JUDGES[judge]
    inp = build_judge_input(record)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if judge == "gpt5":
                sc = call_gpt5(inp)
            elif judge == "claude_sonnet_37":
                sc = call_claude(inp, cfg["model"])
            else:
                sc = call_gemini(inp, cfg["model"])
            result = {"name": record["name"], "date": record["date"],
                      "judge": judge, "label": cfg["label"],
                      "attempt": attempt, "scores": sc.model_dump(by_alias=True)}
            path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
            return result
        except Exception as exc:
            print(f"      [retry {attempt}] {type(exc).__name__}: {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(3 * attempt)
    return None


def flatten(r: dict[str, Any], sr: bool, wf: str) -> dict[str, Any]:
    row = {"source_reflection": sr, "workflow": wf,
           "sample_name": r["name"], "analysis_date": r["date"],
           "judge": r["judge"], "judge_label": r["label"]}
    nums = []
    for dim in DIMENSIONS:
        key = dim.lower().replace("-","_")
        val = r["scores"][dim]
        score = int(val["Score"]) if isinstance(val,dict) else int(val)
        row[f"{key}_score"] = score
        if isinstance(val, dict):
            row[f"{key}_justification"] = val.get("Justification","")
        nums.append(score)
    row["mean_score"] = sum(nums) / len(nums)
    return row

print("Judge clients ready.")

In [ ]:
# ── Run all conditions × all judges ──────────────────────────────────────────
# GPT-5: loaded from disk (no API calls). Claude + Gemini: computed and cached.

all_rows: list[dict[str, Any]] = []

for sr, wf in CONDITIONS:
    label = f"workflow_{sr}/{wf}"
    print(f"\n{'─'*55}\n{label}")
    records = build_records(sr, wf)
    print(f"  {len(records)} records")

    for judge, cfg in JUDGES.items():
        if not cfg["enabled"]:
            print(f"  [{cfg['label']}] SKIPPED"); continue
        print(f"  [{cfg['label']}]")

        if judge == "gpt5":
            existing = load_gpt5_existing(sr, wf)
            if existing:
                # mirror into multimodel folder
                d = judge_dir("gpt5", sr, wf); d.mkdir(parents=True, exist_ok=True)
                for r in existing:
                    p = d / f"{safe_slug(r['name'])}.json"
                    if not p.exists() or OVERWRITE:
                        p.write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding="utf-8")
                print(f"    {len(existing)} existing results loaded (no API calls)")
                all_rows.extend(flatten(r, sr, wf) for r in existing)
                continue

        for i, rec in enumerate(records, 1):
            res = run_judge(rec, judge, sr, wf)
            ok  = "✓" if res else "✗"
            print(f"    [{i:02d}/{len(records)}] {rec['name']} {ok}")
            if res:
                all_rows.append(flatten(res, sr, wf))

results_df = pd.DataFrame(all_rows)
results_df.to_csv(OUT_ROOT / "all_results_raw.csv", index=False)

summary = (results_df.groupby(["source_reflection","workflow","judge_label"])
           .agg(n=("sample_name","count"), mean=("mean_score","mean"))
           .round(3))
display(summary)

In [ ]:
# ── Comparison table, ensemble scores, and paired t-tests ────────────────────

def bci(scores, n=5000, ci=0.95):
    means = [np.random.choice(scores, len(scores), replace=True).mean() for _ in range(n)]
    a = (1-ci)/2
    return float(np.percentile(means, a*100)), float(np.percentile(means, (1-a)*100))


# Per-judge summary
pj_rows = []
for (sr,wf,jn), grp in results_df.groupby(["source_reflection","workflow","judge"]):
    row = {"reflection":str(sr), "workflow":wf, "judge":JUDGES[jn]["label"], "n":len(grp)}
    for col in SCORE_COLS:
        s = grp[col].dropna().values; lo,hi = bci(s)
        row[col]=s.mean(); row[f"{col}_lo"]=lo; row[f"{col}_hi"]=hi
    s=grp["mean_score"].dropna().values; lo,hi=bci(s)
    row["mean"]=s.mean(); row["mean_lo"]=lo; row["mean_hi"]=hi
    pj_rows.append(row)
pj_df = pd.DataFrame(pj_rows)
pj_df.to_csv(OUT_ROOT/"summary_per_judge.csv", index=False)

print("=== PER-JUDGE MEAN SCORES (95% CI) ===\n")
disp = pj_df[["reflection","workflow","judge","n"]].copy()
for col in SCORE_COLS:
    short = col.replace("_score","")
    disp[short] = pj_df.apply(lambda r: f"{r[col]:.2f} [{r[f'{col}_lo']:.2f},{r[f'{col}_hi']:.2f}]", axis=1)
disp["mean"] = pj_df.apply(lambda r: f"{r['mean']:.3f} [{r['mean_lo']:.3f},{r['mean_hi']:.3f}]", axis=1)
display(disp)

# Ensemble (average across judges per sample)
ens = (results_df.groupby(["source_reflection","workflow","sample_name","analysis_date"],
                           as_index=False)[SCORE_COLS].mean())
ens["mean_score"] = ens[SCORE_COLS].mean(axis=1)

ens_rows = []
for (sr,wf), grp in ens.groupby(["source_reflection","workflow"]):
    row = {"reflection":str(sr), "workflow":wf, "n":len(grp)}
    for col in SCORE_COLS:
        s=grp[col].dropna().values; lo,hi=bci(s)
        row[col]=s.mean(); row[f"{col}_lo"]=lo; row[f"{col}_hi"]=hi
    s=grp["mean_score"].dropna().values; lo,hi=bci(s)
    row["mean"]=s.mean(); row["mean_lo"]=lo; row["mean_hi"]=hi
    ens_rows.append(row)
ens_df = pd.DataFrame(ens_rows)
ens_df.to_csv(OUT_ROOT/"summary_ensemble.csv", index=False)

print("\n=== ENSEMBLE MEAN (averaged across judges) ===\n")
ed = ens_df[["reflection","workflow","n"]].copy()
for col in SCORE_COLS:
    ed[col.replace("_score","")] = ens_df[col].round(3)
ed["mean"] = ens_df.apply(lambda r: f"{r['mean']:.3f} [{r['mean_lo']:.3f},{r['mean_hi']:.3f}]", axis=1)
display(ed)

# Paired t-tests on ensemble
test_pairs = [
    ((False,"e2e"),    (False,"default"), "e2e vs default (no reflection)"),
    ((True, "e2e"),    (True, "default"), "e2e vs default (with reflection)"),
    ((True, "default"),(False,"default"), "default: reflection vs no reflection"),
    ((True, "e2e"),    (False,"e2e"),     "e2e: reflection vs no reflection"),
    ((False,"e2e"),    (True, "default"), "best e2e vs best default"),
]
trows = []
all_cols = SCORE_COLS + ["mean_score"]
for (sr_a,wf_a),(sr_b,wf_b),lbl in test_pairs:
    A = ens[(ens["source_reflection"]==sr_a)&(ens["workflow"]==wf_a)]
    B = ens[(ens["source_reflection"]==sr_b)&(ens["workflow"]==wf_b)]
    shared = set(A["sample_name"]) & set(B["sample_name"])
    if len(shared)<3: continue
    A = A[A["sample_name"].isin(shared)].sort_values("sample_name")
    B = B[B["sample_name"].isin(shared)].sort_values("sample_name")
    for col in all_cols:
        a,b = A[col].values, B[col].values
        try: t,p = scipy_stats.ttest_rel(a,b)
        except: t,p = float("nan"),float("nan")
        trows.append({"comparison":lbl,"dimension":col,
                      "mean_A":round(a.mean(),3),"mean_B":round(b.mean(),3),
                      "diff":round((a-b).mean(),3),"t":round(t,3),
                      "p":round(p,4),"sig_p05":p<0.05})
tt_df = pd.DataFrame(trows)
tt_df.to_csv(OUT_ROOT/"paired_ttests.csv", index=False)

print("\n=== PAIRED T-TESTS ===\n")
display(tt_df)
sig = tt_df[tt_df["sig_p05"]]
print("\nSignificant (p<0.05):" if not sig.empty else "\nNo significant differences (p<0.05).")
if not sig.empty: display(sig[["comparison","dimension","diff","p"]])